# Notebook 01 — The zonally averaged climatology: jets, temperature, and the Hadley cell

**Atmospheric General Circulation · Week 1 · Chapter 1 (the "mini-atlas") and Appendix A**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cstdl-atmos-snu/atmospheric-general-circulation/blob/main/notebooks/01-zonal-mean-climatology.ipynb)

In the Intro lecture we said that the general circulation is studied through the *statistics* of the flow — means, variances, covariances — after decomposing it into a **zonal mean** and **eddies**, and into a **time mean** and **transients** (Appendix A). This notebook builds the first, simplest of those statistics: the climatological, zonally averaged state of the troposphere and lower stratosphere.

You will reproduce three of the "building blocks" from Chapter 1:

1. the zonal-mean zonal wind $[\bar u]$ — the subtropical and eddy-driven jets;
2. the zonal-mean temperature $[\bar T]$ — and its relation to $[\bar u]$ through thermal wind;
3. the mean meridional circulation — the Hadley, Ferrel and polar cells, via the mass streamfunction.

**Notation** (Appendix A): square brackets $[\ ]$ denote a zonal average, an overbar a time average, an asterisk the departure from the zonal mean, a prime the departure from the time mean.

**Data.** Monthly long-term means (1991–2020) of the NCEP/NCAR Reanalysis 1 on a 2.5° grid and 17 pressure levels, read directly from NOAA PSL's OPeNDAP server. It is coarse by today's standards, but it needs no registration and is more than enough for the zonal-mean picture. (Later in the course we will switch to ERA5.)

In [ ]:
# --- Setup: install what is missing (only needed on Google Colab) ---
import importlib, subprocess, sys
for pkg, mod in [("xarray", "xarray"), ("netcdf4", "netCDF4"), ("cartopy", "cartopy"), ("cmocean", "cmocean")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cmocean
print("xarray", xr.__version__)

In [ ]:
# --- Helpers (same as scripts/agc.py in the repository, copied here so the notebook is self-contained) ---
PSL = "https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis.derived"
PSL_HTTP = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.derived"

def open_ncep_ltm(var, level_type="pressure", period="1991-2020"):
    """Monthly long-term-mean climatology of an NCEP/NCAR R1 variable (OPeNDAP, with HTTP fallback)."""
    fname = f"{var}.mon.ltm.{period}.nc"
    try:
        ds = xr.open_dataset(f"{PSL}/{level_type}/{fname}")
    except OSError:
        import os, urllib.request
        os.makedirs("data", exist_ok=True)
        local = os.path.join("data", fname)
        if not os.path.exists(local):
            print("OPeNDAP failed; downloading", fname)
            urllib.request.urlretrieve(f"{PSL_HTTP}/{level_type}/{fname}", local)
        ds = xr.open_dataset(local)
    if "time" in ds.dims and ds.sizes["time"] == 12:   # dummy year -> month number
        ds = ds.assign_coords(time=np.arange(1, 13)).rename(time="month")
    return ds

SEASONS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}
def season_mean(da, season):
    return da.sel(month=SEASONS[season]).mean("month")

## 1. Load the data

Three variables on pressure levels: zonal wind `uwnd`, meridional wind `vwnd`, temperature `air`. Each file is 12 months × 17 levels × 73 latitudes × 144 longitudes.

In [ ]:
u = open_ncep_ltm("uwnd")["uwnd"]      # m/s
v = open_ncep_ltm("vwnd")["vwnd"]      # m/s
T = open_ncep_ltm("air")["air"]        # degC in this file
if T.attrs.get("units", "").startswith("deg"):
    T = T + 273.15
    T.attrs["units"] = "K"
u

## 2. Zonal-mean zonal wind $[\bar u]$

Take the zonal average (Appendix A: $[A] = \frac{1}{2\pi}\int_0^{2\pi} A\, d\lambda$) and look at DJF and JJA meridional cross-sections. Compare with the corresponding panels in Chapter 1. Where are the subtropical jets? How does their strength and latitude change between the seasons? Is there a sign that the tropospheric jet in midlatitudes is a separate, "eddy-driven" feature?

In [ ]:
uz = u.mean("lon")   # [u](month, level, lat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True, constrained_layout=True)
levels = np.arange(-50, 51, 5)
for ax, season in zip(axes, ["DJF", "JJA"]):
    field = season_mean(uz, season)
    cf = ax.contourf(field.lat, field.level, field, levels=levels, cmap="RdBu_r", extend="both")
    ax.contour(field.lat, field.level, field, levels=levels, colors="k", linewidths=0.4)
    ax.set_title(f"[u]  {season}")
    ax.set_xlabel("latitude")
    ax.set_xticks(np.arange(-90, 91, 30))
ax = axes[0]; ax.set_yscale("log"); ax.invert_yaxis()
ax.set_yticks([1000, 700, 500, 300, 200, 100, 50, 20, 10]); ax.set_yticklabels([1000, 700, 500, 300, 200, 100, 50, 20, 10])
ax.set_ylabel("pressure (hPa)")
fig.colorbar(cf, ax=axes, label="m s$^{-1}$", shrink=0.9)
plt.show()

## 3. Zonal-mean temperature $[\bar T]$ and thermal wind

Plot $[\bar T]$ for the same seasons. Then check the thermal-wind relation on the sphere,

$$ f\,\frac{\partial [u]}{\partial \ln p} = \frac{R}{a}\,\frac{\partial [T]}{\partial \phi}, $$

by computing the right-hand side from $[\bar T]$ and comparing with the left-hand side from $[\bar u]$. Where does the balance hold well, and where (and why) does it fail?

In [ ]:
Tz = T.mean("lon")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True, constrained_layout=True)
levels = np.arange(190, 311, 5)
for ax, season in zip(axes, ["DJF", "JJA"]):
    field = season_mean(Tz, season)
    cf = ax.contourf(field.lat, field.level, field, levels=levels, cmap=cmocean.cm.thermal, extend="both")
    ax.contour(field.lat, field.level, field, levels=levels[::2], colors="k", linewidths=0.4)
    ax.set_title(f"[T]  {season}"); ax.set_xlabel("latitude"); ax.set_xticks(np.arange(-90, 91, 30))
ax = axes[0]; ax.set_yscale("log"); ax.invert_yaxis()
ax.set_yticks([1000, 700, 500, 300, 200, 100, 50, 20, 10]); ax.set_yticklabels([1000, 700, 500, 300, 200, 100, 50, 20, 10])
ax.set_ylabel("pressure (hPa)")
fig.colorbar(cf, ax=axes, label="K", shrink=0.9)
plt.show()

In [ ]:
# Thermal wind check (DJF)
a, R, Omega = 6.371e6, 287.0, 7.292e-5
phi = np.deg2rad(uz.lat)
f = 2 * Omega * np.sin(phi)

uDJF, TDJF = season_mean(uz, "DJF"), season_mean(Tz, "DJF")
lnp = np.log(uz.level)
lhs = (uDJF.differentiate("level") * uz.level * f).transpose("level", "lat")   # f du/d(ln p) = f p du/dp
rhs = ((R / a) * TDJF.differentiate("lat") * (180 / np.pi)).transpose("level", "lat")  # (R/a) dT/dphi, lat in degrees -> radians

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True, constrained_layout=True)
levels = np.linspace(-1.5e-3, 1.5e-3, 16)
for ax, fld, ttl in zip(axes, [lhs, rhs], ["f ∂[u]/∂ln p", "(R/a) ∂[T]/∂φ"]):
    cf = ax.contourf(fld.lat, fld.level, fld, levels=levels, cmap="PuOr_r", extend="both")
    ax.set_title(ttl + "  (DJF)"); ax.set_xlabel("latitude"); ax.set_xticks(np.arange(-90, 91, 30))
ax = axes[0]; ax.set_yscale("log"); ax.invert_yaxis(); ax.set_ylim(1000, 100); ax.set_ylabel("pressure (hPa)")
fig.colorbar(cf, ax=axes, label="m s$^{-2}$", shrink=0.9)
plt.show()

## 4. The mean meridional circulation

The zonally averaged flow in the meridional plane is best displayed with the **mass streamfunction**. With $[\bar v]$ in pressure coordinates,

$$ \Psi(\phi, p) = \frac{2\pi a \cos\phi}{g} \int_0^{p} [\bar v]\, dp', $$

so that $[\bar v] = \frac{g}{2\pi a\cos\phi}\,\partial\Psi/\partial p$. Positive $\Psi$ means clockwise circulation in a plot with the equator on the left... check the sign convention yourself with the plot. Identify the Hadley, Ferrel and polar cells, and note how strongly the Hadley circulation is dominated by the *winter* hemisphere cell.

In [ ]:
def mass_streamfunction(vz, a=6.371e6, g=9.81):
    """Psi(month, level, lat) in kg/s from zonal-mean v(month, level, lat) on pressure levels (hPa)."""
    vz = vz.sortby("level")                      # top (low p) -> bottom
    p = vz.level * 100.0                         # Pa
    dp = p.diff("level")
    vmid = 0.5 * (vz.isel(level=slice(1, None)).values + vz.isel(level=slice(None, -1)).values)
    integrand = vmid * dp.values[None, :, None]  # (month, level-1, lat)
    psi = np.cumsum(integrand, axis=1)
    psi = np.concatenate([np.zeros_like(psi[:, :1]), psi], axis=1)  # Psi = 0 at the top
    coslat = np.cos(np.deg2rad(vz.lat.values))[None, None, :]
    psi = 2 * np.pi * a * coslat / g * psi
    return xr.DataArray(psi, coords=vz.coords, dims=vz.dims, name="psi", attrs={"units": "kg s-1"})

vz = v.mean("lon")
psi = mass_streamfunction(vz) / 1e10   # 10^10 kg/s

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, constrained_layout=True)
levels = np.arange(-20, 21, 2)
for ax, season in zip(axes, ["DJF", "JJA", "annual"]):
    field = psi.mean("month") if season == "annual" else season_mean(psi, season)
    cf = ax.contourf(field.lat, field.level, field, levels=levels, cmap="RdBu_r", extend="both")
    ax.contour(field.lat, field.level, field, levels=levels, colors="k", linewidths=0.4)
    ax.set_title(f"Ψ  {season}"); ax.set_xlabel("latitude"); ax.set_xticks(np.arange(-90, 91, 30))
ax = axes[0]; ax.invert_yaxis(); ax.set_ylim(1000, 100); ax.set_ylabel("pressure (hPa)")
fig.colorbar(cf, ax=axes, label="10$^{10}$ kg s$^{-1}$", shrink=0.9)
plt.show()

## 5. Beyond the zonal mean: the 250 hPa jet streams

Everything above treated the Earth as an aquaplanet. As a preview of Part V, map the DJF climatological $\bar u$ at 250 hPa. The zonal mean hides that the jet is concentrated over the western Pacific and western Atlantic.

In [ ]:
import cartopy.crs as ccrs
u250 = season_mean(u.sel(level=250), "DJF")
# wrap the longitude so the map has no seam at 0°E
u250 = xr.concat([u250, u250.isel(lon=0).assign_coords(lon=360.0)], dim="lon")

fig = plt.figure(figsize=(12, 5))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
cf = ax.contourf(u250.lon, u250.lat, u250, levels=np.arange(-30, 71, 5), cmap="RdBu_r", extend="both", transform=ccrs.PlateCarree())
ax.coastlines(linewidth=0.6); ax.set_global()
ax.set_title("DJF climatological u at 250 hPa")
fig.colorbar(cf, ax=ax, orientation="horizontal", shrink=0.6, pad=0.05, label="m s$^{-1}$")
plt.show()

## Exercises (bring your figures to class)

1. **Seasonal march.** Plot the latitude of the maximum $[\bar u]$ at 200 hPa in each hemisphere as a function of month. How far does the subtropical jet migrate? Compare with the migration of the Hadley cell boundary (the latitude where $\Psi$ at 500 hPa changes sign).
2. **Thermal wind.** Quantify how well thermal wind balance holds by plotting the ratio (or difference) of the two panels of Section 3 as a function of latitude at 500 hPa. Where does geostrophy break down, and what replaces it?
3. **Hadley cell strength.** Read off the maximum $|\Psi|$ of the DJF and JJA Hadley cells. Which is stronger, and why might the winter cell dominate? (Chapter 1 and, later, Chapters 3 and 7.)
4. **\*** *(open-ended)* Repeat Section 2 for a single year rather than the 1991–2020 climatology (the PSL server also serves monthly means: `.../pressure/uwnd.mon.mean.nc`). How different is one winter from the climatology? This is your first look at the "transients" of Appendix A.